[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tasmia008/Complete_Guidelines_LLM_FineTuning/blob/main/08_instruction_tuning.ipynb)

# Instruction tuning

> **Fine-tuning series — 08 of 26.** A standalone notebook in a set; see `00_README.md` for the full index and order. This notebook is self-contained: run **Setup**, then the rest.


## Setup (run first)

Self-contained: imports, model id, device flags, and a tiny inline dataset so this notebook runs on its own.

In [ ]:
# Environment check — record these versions with any results you report.
import importlib
for lib in ["torch", "transformers", "peft", "trl", "datasets",
            "bitsandbytes", "accelerate", "unsloth", "adapters"]:
    try:
        m = importlib.import_module(lib)
        print(f"{lib:14s} {getattr(m, '__version__', '?')}")
    except Exception as e:
        print(f"{lib:14s} not installed ({type(e).__name__})")

In [ ]:
# !pip install -U transformers peft trl bitsandbytes datasets accelerate
# !pip install unsloth   # only for the Unsloth section (CUDA only)
import gc, torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"     # small + fast; swap for any causal LM
device = ("cuda" if torch.cuda.is_available()
          else "mps" if torch.backends.mps.is_available() else "cpu")
BF16_OK = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
FP16_OK = torch.cuda.is_available() and not BF16_OK   # fp16 needs a GPU
print("device:", device, "| bf16:", BF16_OK)

def cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
# (a) Instruction data -> for SFT / LoRA / QLoRA / Unsloth / instruction / prompt tuning
instructions = [
    {"instruction": "Give the capital of France.", "output": "Paris."},
    {"instruction": "What is 7 times 8?", "output": "56."},
    {"instruction": "Translate 'good morning' to Spanish.", "output": "Buenos días."},
    {"instruction": "Name the largest planet.", "output": "Jupiter."},
    {"instruction": "Define photosynthesis in one line.",
     "output": "Plants converting sunlight, water, and CO2 into energy and oxygen."},
] * 8   # repeat just to have enough steps in the demo

# (b) Preference data -> for DPO (prompt + chosen/rejected, no reward model)
preferences = [
    {"prompt": "Explain gravity to a child.",
     "chosen": "Gravity is the pull that brings things down to the ground.",
     "rejected": "Gravity is a tensor field obeying the Einstein equations."},
    {"prompt": "Recommend a book.",
     "chosen": "Try 'The Hobbit' — a fun, easy adventure.",
     "rejected": "Read whatever, I don't care."},
] * 16

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def to_chat_text(ex):
    msgs = [{"role": "user", "content": ex["instruction"]},
            {"role": "assistant", "content": ex["output"]}]
    return {"text": tokenizer.apply_chat_template(msgs, tokenize=False)}

sft_ds = Dataset.from_list([to_chat_text(e) for e in instructions])
pref_ds = Dataset.from_list(preferences)
print(sft_ds[0]["text"][:160])

In [ ]:
# Shared trainer imports + a default LoRA config reused by several sections.
from trl import SFTTrainer, SFTConfig
from peft import LoraConfig
lora_cfg = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"])

## 5. Instruction tuning  ·  *what-axis*

This **is** supervised fine-tuning — on (instruction → response) pairs in chat
format. The one thing that makes it "instruction tuning done right": compute loss
**only on the assistant's response**, not the prompt — done by setting the prompt's
label positions to `-100` (the ignore index from notebook 01 §5). TRL used to ship
`DataCollatorForCompletionOnlyLM` for this, but it was **removed in TRL ≥ 1.0**, so the
cell below masks the labels directly: version-proof, and it shows exactly what the
frameworks automate. (Combine freely with any how-axis method above — here it's LoRA.)

In [ ]:
# Loss only on the assistant response. The collator that used to do this
# (DataCollatorForCompletionOnlyLM) was removed in TRL >= 1.0, so we mask the labels
# ourselves — works on any version — and train with a plain Trainer + LoRA.
from transformers import Trainer, TrainingArguments
from peft import get_peft_model

def mask_to_response(ex):
    msgs = [{"role": "user", "content": ex["instruction"]},
            {"role": "assistant", "content": ex["output"]}]
    ids = tokenizer.apply_chat_template(msgs, tokenize=True, return_dict=True)["input_ids"]
    # everything up to where the assistant turn opens is prompt -> mask it to -100
    prompt_len = len(tokenizer.apply_chat_template(
        msgs[:1], tokenize=True, add_generation_prompt=True, return_dict=True)["input_ids"])
    labels = [-100] * prompt_len + ids[prompt_len:]
    return {"input_ids": ids, "labels": labels}

masked_ds = Dataset.from_list([mask_to_response(e) for e in instructions])
print("loss is computed only on:",
      repr(tokenizer.decode([t for t in masked_ds[0]["labels"] if t != -100])))

def pad_collate(batch):                       # pad input_ids with pad_token, labels with -100
    maxlen = max(len(b["input_ids"]) for b in batch)
    pad = tokenizer.pad_token_id
    out = {"input_ids": [], "attention_mask": [], "labels": []}
    for b in batch:
        n = maxlen - len(b["input_ids"])
        out["input_ids"].append(b["input_ids"] + [pad] * n)
        out["attention_mask"].append([1] * len(b["input_ids"]) + [0] * n)
        out["labels"].append(b["labels"] + [-100] * n)
    return {k: torch.tensor(v) for k, v in out.items()}

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16 if BF16_OK else torch.float32).to(device)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()            # confirm <1%, not 0%

trainer = Trainer(
    model=model, train_dataset=masked_ds, data_collator=pad_collate,
    args=TrainingArguments(output_dir="ft_instruct",
                   per_device_train_batch_size=2, max_steps=20,
                   learning_rate=2e-4, logging_steps=5,
                   bf16=BF16_OK, report_to="none"))
trainer.train()
del trainer, model; cleanup()

### Multi-turn masking — *which* assistant turns get loss

The cell above masks one assistant turn. A multi-turn conversation
(`user Q1 → asst A1 → user Q2 → asst A2`) has several, and raises two questions:

1. **Correctness** — do the in-between *user* turns leak into the loss?
2. **Design** — train on *all* assistant turns, or only the *last* one?

`build_labels` below generalizes the single-turn masker above to any number of turns,
using only the tokenizer — so it runs on any TRL version and shows exactly what the
framework flags (`assistant_only_loss`, `train_on_responses_only`) automate.

In [ ]:
# Build the label mask by hand — only needs the tokenizer, so it is version-proof.
# Rule: a position trains (label = token id) only inside an assistant turn; everything
# else is masked to -100 (the ignore index from notebook 01, section 5).
chat = [
    {"role": "user",      "content": "What's the capital of France?"},
    {"role": "assistant", "content": "Paris."},
    {"role": "user",      "content": "And of Japan?"},
    {"role": "assistant", "content": "Tokyo."},
]

def ids_of(messages, add_gen=False):
    return tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=add_gen, return_dict=True)["input_ids"]

def build_labels(messages, last_only=False):
    ids = ids_of(messages)
    labels = [-100] * len(ids)
    asst = [i for i, m in enumerate(messages) if m["role"] == "assistant"]
    for i in (asst[-1:] if last_only else asst):           # which turns to train on
        start = len(ids_of(messages[:i], add_gen=True))    # right where asst content begins
        end   = len(ids_of(messages[:i + 1]))              # through its closing <|im_end|>
        labels[start:end] = ids[start:end]                 # unmask just this assistant span
    return ids, labels

def trained_text(messages, last_only=False):
    ids, labels = build_labels(messages, last_only)
    return tokenizer.decode([t for t in labels if t != -100])

In [ ]:
print("ALL assistant turns >>>", repr(trained_text(chat)))
print("LAST turn only      >>>", repr(trained_text(chat, last_only=True)))

ids, labels = build_labels(chat)
print(f"\n{sum(l != -100 for l in labels)} of {len(ids)} tokens train; "
      "the user turns ('And of Japan?') are masked, the closing <|im_end|> is kept "
      "(so the model learns to stop).")

**The trap.** The naive single-turn recipe — one `response_template`, unmask everything
after it — keeps going past the first answer and trains on the *second user turn*, teaching
the model to generate user questions. Masking each assistant span individually (above) is
what avoids it.

**All vs. last.** Training on **all** assistant turns is the default and most
data-efficient — correct when every turn is gold quality. Train on the **last** turn only
(`last_only=True`) when earlier turns are context from a weaker model, or when only the
final answer passed a verifier (rejection sampling, notebook 14).

**How the frameworks do it for you** (all equivalent to `build_labels` above):

| Tool | Call | Caveat |
|---|---|---|
| TRL (current) | `SFTConfig(assistant_only_loss=True)` | needs a chat template with `{% generation %}` tags — many lack them (checked below) |
| TRL (≤ 0.x) | `DataCollatorForCompletionOnlyLM(instruction_template, response_template)` | **removed in TRL ≥ 1.0**; pass *both* templates for multi-turn |
| Unsloth | `train_on_responses_only(trainer, instruction_part, response_part)` | works on any template |

When the template has no generation tags and the collator is gone, the manual masker above
is the portable fallback — feed its `labels` to a plain `Trainer`.

In [ ]:
# Does THIS model's chat template support the assistant_only_loss flag?
out = tokenizer.apply_chat_template(
    chat, return_assistant_tokens_mask=True, return_dict=True, tokenize=True)
mask = out.get("assistant_masks") or []
if sum(mask):
    kept = tokenizer.decode([i for i, m in zip(out["input_ids"], mask) if m])
    print("Template HAS generation tags -> assistant_only_loss trains on:", repr(kept))
else:
    print("Template has NO {% generation %} tags (Qwen2.5 is one such case):")
    print("  -> assistant_only_loss would mask everything; use build_labels() above instead.")